# Fairness in recommender systems
## Part 1 · Worked audit of an internship board

**Question:** Can we give small employers more visibility while preserving a useful top-3 list for students?

This fictional case has four learners (N1/N2: new; R1/R2: returning) and four eligible listings. A and B come from large employers; C and D come from small employers. The four groups describe activity history or employer size. They are distinct dimensions and not claims about protected groups. No real people, employers or hiring decisions are represented.

**Start:** Extract the lab ZIP and open this notebook from the extracted folder or its `notebooks` folder. Only the Python standard library is used. Run each section in order. Pause before printing: predict the numbers for N1 first.

- `listings.csv`: listing ID, employer size, role (all listings are eligible in the main case).
- `learners.csv`: learner ID, activity group, fixed baseline order of all four items.
- `relevance.csv`: offline grades for all learner–item pairs, 0 = poor fit, 1 = plausible, 2 = strong fit. The grades are separate from the baseline ranks. In practice one needs defensible held-out judgments.

In [1]:
from pathlib import Path  # Locate the extracted lab.
import sys  # Make the companion module importable.

here = Path.cwd().resolve()  # Start from the notebook's working folder.
root = next(p for p in (here, *here.parents) if (p / "src/audit.py").is_file())  # Find the lab root.
sys.path.insert(0, str(root / "src"))  # Load the module from this copy of the lab.
from audit import load_case, dcg, ndcg, exposure, promote_small, compare  # Reuse auditable helpers.

size, segments, baseline, grades = load_case()  # Read the three CSV files.
for u, ranking in baseline.items():  # One request per learner in this exercise.
    print(u, segments[u], "baseline:", ranking, "judgments:", grades[u])  # Compare inputs.

N1 New baseline: ['A', 'B', 'C', 'D'] judgments: {'A': 2, 'B': 0, 'C': 2, 'D': 1}
N2 New baseline: ['B', 'A', 'D', 'C'] judgments: {'A': 0, 'B': 2, 'C': 1, 'D': 2}
R1 Returning baseline: ['A', 'B', 'C', 'D'] judgments: {'A': 2, 'B': 2, 'C': 1, 'D': 0}
R2 Returning baseline: ['B', 'A', 'D', 'C'] judgments: {'A': 2, 'B': 2, 'C': 0, 'D': 1}


### 1. User quality: NDCG@3

The algorithm displays three items. N1 sees A, B, C with grades 2, 0, 2. Grades map to gains via `2**grade - 1`: 2 → 3, 1 → 1, 0 → 0. Multiply each gain by `1 / log2(r + 1)`, where rank r starts at 1. For the first three positions, these discounts are 1, 0.6309 and 0.5. The DCG of the displayed list is `3 + 0 + 1.5 = 4.5`.

Order **all four eligible judged items** by grade to calculate the ideal DCG. The ideal top-3 grades are 2, 2, 1, giving 5.3928. Thus N1's NDCG@3 is 4.5/5.3928 ≈ 0.834. An unshown but known relevant item can matter to the ideal denominator.

In [2]:
u = "N1"  # Trace one learner before aggregating.
shown = baseline[u][:3]  # Take the first three positions.
shown_grades = [grades[u][item] for item in shown]  # Read independent judgments.
ideal_grades = sorted(grades[u].values(), reverse=True)[:3]  # Best from all eligible items.
print("shown:", shown, "grades:", shown_grades, "DCG:", round(dcg(shown_grades), 4))  # Numerator.
print("ideal grades:", ideal_grades, "IDCG:", round(dcg(ideal_grades), 4))  # Denominator.
print("NDCG:", round(ndcg(baseline[u], grades[u]), 3))  # Ratio.

shown: ['A', 'B', 'C'] grades: [2, 0, 2] DCG: 4.5
ideal grades: [2, 2, 1] IDCG: 5.3928
NDCG: 0.834


Calculate one NDCG per learner, then compare group means and the number of learners covered. Here every ideal DCG is positive. With an all-zero judged pool, the ratio is undefined: mark it missing and report the excluded count. An unjudged item is different from a judged grade zero.

In [3]:
from statistics import mean  # Average first across learners, not across list positions.

quality = {u: ndcg(items, grades[u]) for u, items in baseline.items()}  # Score each learner.
for group in ("New", "Returning"):  # Compare both learner groups.
    values = [v for u, v in quality.items() if segments[u] == group and v is not None]  # Defined values.
    print(group, "n =", len(values), "NDCG@3 =", round(mean(values), 3))  # Keep counts visible.

New n = 2 NDCG@3 = 0.834
Returning n = 2 NDCG@3 = 1.0


### 2. Provider visibility: exposure

Use a **separate, illustrative attention model**: weight 1 at rank 1, 0.5 at rank 2, 0.25 at rank 3 and zero otherwise. These are neither observed clicks nor the DCG discounts. Each of four lists contributes 1.75 units, giving 7 total.

C and D each receive 0.5 units across the four lists. Small listings therefore receive 1/7 ≈ 14.3% of attention, although they occupy 4/12 ≈ 33.3% of slots. Both groups have two listings in this fixture. In larger audits also report exposure per eligible item and how listings get into the candidate pool.

What target would you defend? Equal exposure share is one option in this balanced example. Relevance-proportional exposure is another: the small listings have 8 out of 20 rubric grade points over all eligible pairs, so that **illustrative** target is 40%. The target depends on the chosen eligibility criteria, relevance definition and normative decision.

In [4]:
weights = (1, 0.5, 0.25)  # Simplified attention model from class.
for item, weight in zip(baseline["N1"][:3], weights):  # Pair item and position weight.
    print(item, size[item], weight)  # Trace N1's allocation.
by_item = exposure(baseline, size)  # Sum over all four ranked lists.
small = by_item["C"] + by_item["D"]  # Combine the two small listings.
print(by_item)  # Show where visibility goes.
print("small share =", f"{small / sum(by_item.values()):.1%}")  # Divide by all attention.

A Large 1
B Large 0.5
C Small 0.25
{'A': 3.0, 'B': 3.0, 'C': 0.5, 'D': 0.5}
small share = 14.3%


### 3. Post-processing: move an eligible small listing to rank 2

If the first two places have no small listing, move the highest-ranked small candidate to position 2. Keep the others in their original relative order. If none are eligible in the candidate list, skip the change. The rule reads the baseline order and employer group. It does not peek at the audit judgments.

N1 goes from A, B, C, D to A, C, B, D. C moves up while B moves down. N1 benefits because C is a better fit according to the independent judgment. R1 has the same baseline order but different judgments, so moving C above B reduces R1's NDCG.

In [5]:
reranked = {u: promote_small(items, size, target=2) for u, items in baseline.items()}  # Rerank.
for u in baseline:  # Check the same learners before and after.
    print(u, baseline[u][:3], "→", reranked[u][:3])  # Only the first three are served.
    print("  NDCG:", f"{ndcg(baseline[u], grades[u]):.3f}", "→",
          f"{ndcg(reranked[u], grades[u]):.3f}")  # Report the changed utility.

N1 ['A', 'B', 'C'] → ['A', 'C', 'B']
  NDCG: 0.834 → 0.907
N2 ['B', 'A', 'D'] → ['B', 'D', 'A']
  NDCG: 0.834 → 0.907
R1 ['A', 'B', 'C'] → ['A', 'C', 'B']
  NDCG: 1.000 → 0.951
R2 ['B', 'A', 'D'] → ['B', 'D', 'A']
  NDCG: 1.000 → 0.951


In [6]:
before = compare("Baseline", baseline, size, segments, grades)  # Audit the fixed baseline.
after = compare("Promote to 2", reranked, size, segments, grades)  # Keep the same test cohort.
for result in (before, after):  # Report both sides of the change.
    print(result["policy"], f'new={result["new_ndcg"]:.3f}',
          f'returning={result["returning_ndcg"]:.3f}', f'overall={result["overall_ndcg"]:.3f}',
          f'small exposure={result["small_exposure_share"]:.1%}')  # Use explicit units.

Baseline new=0.834 returning=1.000 overall=0.917 small exposure=14.3%
Promote to 2 new=0.907 returning=0.951 overall=0.929 small exposure=28.6%


**Read both sides:** Small exposure rises from 14.3% to 28.6%. Mean NDCG improves for new learners from 0.834 to 0.907. It falls for returning learners from 1.000 to 0.951. The overall mean rises from 0.917 to 0.929. The policy is an experiment, not a universal recommendation: the four learners are too few to estimate production impact.

### 4. Which stage loses the opportunity?

Now suppose retrieval drops C for N1. Use grade > 0 to define a relevant candidate. There are three relevant eligible items (A, C, D); two survive (A, D). Candidate recall is 2/3. Reranking can move D, but it cannot retrieve C. If you redefine the ideal reference to only surviving candidates, you answer a different, conditional question rather than evaluating end-to-end loss.

In [7]:
full = baseline["N1"]  # Original eligible candidate list.
retrieved = [item for item in full if item != "C"]  # Simulate a retrieval failure.
relevant = {item for item, grade in grades["N1"].items() if grade > 0}  # Declare relevance.
hits = relevant.intersection(retrieved)  # Count relevant eligible candidates retrieved.
print("candidate recall:", len(hits), "/", len(relevant))  # Expected: 2 / 3.
print("new top 3:", promote_small(retrieved, size, target=2)[:3])  # C is still absent.

candidate recall: 2 / 3
new top 3: ['A', 'D', 'B']


Clicks require exposure. A vacancy never shown has no chance to be clicked in that session. If we train only on those clicks, a ranking based on popularity can favor items that already receive attention. For auditing, log the candidate list, displayed ranks and outcomes separately. Use independent judgments or carefully designed exposure experiments when needed; comparisons of observational click rates do not isolate a causal position effect.

## Part 2 · Home study and final project

**TODO A. A stronger reranker.** Run the cell with target 2, then change it to 1. Report each group mean, small exposure, and the overall mean. Suppose the policy owner sets a minimum mean NDCG@3 of 0.94 for returning learners, in advance. Which variants pass? Would you add a minimum standard for new learners too? Explain what these thresholds do and do not protect.

**TODO B. Robustness.** Try exposure weights (1, 0.8, 0.6) and recompute exposure. Compare slot share, attention share and exposure per eligible listing. Try one different mix of learners and explain whether ranking quality and exposure move together.

**TODO C. Retrieval and logging.** Remove both small candidates for one learner. Measure relevant candidate recall and report how often the reranking rule skips. Propose a change at candidate retrieval and compare under the same eligibility rules.

**TODO D. Independent evidence.** Write an evaluation plan for a larger cohort. Specify sampling, judgment quality, relevant segments, per-listing exposure, a utility guardrail, and a rollback rule. Track eligible impressions, applications and interviews separately. Repeated requests from one learner are correlated. Do not treat them as independent people.

In [8]:
target = 2  # TODO A: change to 1 and compare.
experiment = {u: promote_small(items, size, target) for u, items in baseline.items()}  # Apply policy.
report = compare(f"Promote to {target}", experiment, size, segments, grades)  # Calculate outcomes.
floor = 0.94  # Illustrative minimum for returning learners.
print(report)  # Compare with the baseline report.
print("passes returning utility floor:", report["returning_ndcg"] >= floor)  # Check guardrail.

{'policy': 'Promote to 2', 'new_ndcg': 0.9072836011519267, 'returning_ndcg': 0.9514426589871553, 'overall_ndcg': 0.9293631300695411, 'learners_with_ndcg': 4, 'small_slot_share': 0.3333333333333333, 'small_exposure_share': 0.2857142857142857, 'total_exposure': 7.0}
passes returning utility floor: True


**Project submission:** a reproducible notebook with documented data and counts, a comparison of at least two policies, and a short recommendation naming who gains, who loses, and what could change the conclusion. 

**Reading:** [Fairness of Exposure in Rankings](https://arxiv.org/abs/1802.07281), [Unbiased Learning-to-Rank with Biased Feedback](https://doi.org/10.1145/3018661.3018699), and [Equity of Attention](https://arxiv.org/abs/1805.01788).